# Amazon Product Reviews — Sentiment Analysis

**Goal:** Analyze 2,500 public product reviews and group them into Positive, Negative, and Neutral using VADER sentiment analysis.

**Dataset source:** Cleanlab Amazon Reviews dataset on Hugging Face: https://huggingface.co/datasets/Cleanlab/amazon-reviews

**Important:** This notebook uses an existing public dataset; it does not scrape new reviews directly from Amazon. Describe the source honestly in your report.

In [ ]:
!pip -q install vaderSentiment

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Load the public dataset (5,000 training reviews)
url = 'https://huggingface.co/datasets/Cleanlab/amazon-reviews/resolve/main/train.csv'
df = pd.read_csv(url)
print('Available columns:', list(df.columns))
print('Rows available:', len(df))
display(df.head())

In [ ]:
# Keep 2,500 reviews for this project
if 'review_text' not in df.columns:
    raise ValueError('Expected review_text column was not found. Check the dataset page/source URL.')
df = df[['review_text']].dropna()
df['review_text'] = df['review_text'].astype(str).str.strip()
df = df[df['review_text'] != '']
df = df.drop_duplicates(subset=['review_text']).head(2500).copy()

print('Reviews used for analysis:', len(df))

In [ ]:
# Run sentiment analysis and assign three classes
analyzer = SentimentIntensityAnalyzer()
df['compound_score'] = df['review_text'].apply(lambda text: analyzer.polarity_scores(text)['compound'])

def classify_sentiment(score):
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    return 'Neutral'

df['sentiment'] = df['compound_score'].apply(classify_sentiment)
counts = df['sentiment'].value_counts().reindex(['Positive', 'Negative', 'Neutral'], fill_value=0)
summary = pd.DataFrame({'Sentiment': counts.index, 'Count': counts.values})
summary['Percentage'] = (summary['Count'] / len(df) * 100).round(2)
display(summary)
print('Total analyzed reviews:', len(df))

In [ ]:
# Save classified reviews and chart
df.to_csv('analysed_product_reviews.csv', index=False)
plt.figure(figsize=(7, 5))
counts.plot(kind='bar')
plt.title('Sentiment Distribution of Product Reviews')
plt.xlabel('Sentiment')
plt.ylabel('Number of Reviews')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('sentiment_distribution.png', dpi=200)
plt.show()
print('Saved: analysed_product_reviews.csv and sentiment_distribution.png')

In [ ]:
# Optional: download output files to your computer from Google Colab
from google.colab import files
files.download('analysed_product_reviews.csv')
files.download('sentiment_distribution.png')